# MH²M with independently refined conormals

The oscillatory coefficient follows equation (61) of de Barros, Madureira and Valentin, arXiv:2404.16978v3, Section 8.2, with epsilon=1/14. Barros's 2022 LNCC dissertation supplies gamma=1.8 (Equation 4.4, printed page 62) and the quartic forcing (Equation 4.1, page 54). Its Figure 4 also specifies the crisscross fine connectivity and northwest-to-southeast macro diagonals. The dissertation uses epsilon=1/17; this notebook retains the version-3 article's epsilon=1/14. These source relationships do not identify every historical h/H convention.

The main section solves this same heterogeneous equation on a declared coarse diagonal macro mesh, compares its original coefficient equations, and separately refines a classical conforming reference. The later sections display the multilevel numerical campaigns. The classical conforming references have their own refinement series and use the same physical data. The P1 controls use shared finite-element kernels; the P3 reference is independently assembled with DOLFINx/UFL. None is an exact solution.


The main equations execute from source and state their current discretization. The original archived comparisons below execute with `--historical` and require their exact attributed payloads. `--study` runs the complete public current acquisition declared in the catalogue, with its own output provenance.


The PyMHM distribution contains only the library. The first cell explicitly
downloads a checksum-verified companion archive and prepares the declared data
using its local Python helpers. You can inspect these support files in `ROOT`.
Complete studies and historical replay retain their existing opt-in flags.


In [ ]:
from pathlib import Path
import os
import sys
from pymhm.io.workspace import workspace_from_archive

# Download verified support files; this operation does not execute them.
COMPANION_URL = "https://ipes-lncc.github.io/pymhm/downloads/19b8273383a52eeb710d63bac1f4521a7b6783eb6419212b59058a7966e0c79e/70_mh2m_heterogeneous-companion.zip"
COMPANION_SHA256 = "19b8273383a52eeb710d63bac1f4521a7b6783eb6419212b59058a7966e0c79e"
WORKSPACE = Path(
    os.environ.get("PYMHM_WORKSPACE", Path.cwd() / ".pymhm-companions" / COMPANION_SHA256)
)
ROOT = workspace_from_archive(COMPANION_URL, sha256=COMPANION_SHA256, directory=WORKSPACE)
os.environ["PYMHM_WORKSPACE"] = str(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# Explicitly prepare declared data with the downloaded Python helpers.
from scripts.notebook_reproduction import notebook_workspace

ROOT = notebook_workspace("darcy/70_mh2m_heterogeneous.ipynb", directory=ROOT)
root = ROOT
print("Workspace:", ROOT)


import os

HISTORICAL_REPLAY = os.environ.get("PYMHM_NOTEBOOK_HISTORICAL", "0") == "1"
STUDY_REPLAY = os.environ.get("PYMHM_NOTEBOOK_STUDY", "0") == "1"


In [ ]:
from pathlib import Path
import json
import hashlib
import numpy as np
from IPython.display import Image, display
from threadpoolctl import threadpool_limits
from pymhm.fem.traces.interval import FaceSpace, SkeletonSpace
from pymhm.meshes.triangle import TriangleMesh
from pymhm.fem.traces.pressure_2d import PressureTraceSpace


## Same heterogeneous equation, independent interface spaces

The current calculation uses the literal article coefficient with gamma=1.8 and
 epsilon=1/14, the quartic forcing stated above, and zero pressure on all four
sides of the unit square. It uses a southwest–northeast diagonal macro mesh with
four squares per axis, local continuous P1 pressures with refinement 16, and a
continuous P1 Gamma with one segment per macroedge. Lambda consists of broken
P0 polynomials with one or two independent segments. This is a declared coarse
control of the same physical problem; it does not identify the article's full
crisscross discretization or establish its accuracy.

Gamma determines global pressure-trace unknowns, while Lambda determines private
local conormal unknowns. The outward conormal is K grad(p)·n, opposite to the
outward physical Darcy flux. Each calculation is compared with a direct assembly
of its original A/B/C/D rows. That comparison tests condensation in identical
spaces; it is separate from the classical reference below.

One P0 conormal per fine boundary edge can create an alternating kernel for
continuous P1 pressures on an even boundary cycle. Here each conormal segment
contains at least eight fine boundary edges. Assembly orders 8 and 10 are retained
as a quadrature control, without changing spaces or physical data.


## Declare the three-field equations

The application declares the coupled local pressure/conormal saddle and its
global continuous pressure-trace test, using the independently chosen spaces:

$$
\begin{aligned}
A_T&=\begin{bmatrix}K_T&-\mathcal{B}_T\\-\mathcal{B}_T^{\mathsf T}&0\end{bmatrix},
&B_T&=\begin{bmatrix}0\\P_T\end{bmatrix},\\
C_T&=B_T^{\mathsf T}, &D_T&=0, &f_T&=\begin{bmatrix}F_T\\0\end{bmatrix}.
\end{aligned}
$$

Local coefficients are pressure and outward conormal eta=K grad(p).n; its
negative is physical Darcy normal flux. Gamma shares pressure nodes, while
Lambda remains private to each macrocell. The global `Equation` supplies
outward Neumann moments and fixed Gamma pressure values. Pure Neumann data add
the physical integral of the complete reconstructed pressure. Generic assembly
and solve consume these blocks directly. The shared Neumann-map kernel retains
the original boundary-mean basis for inspection and injectivity checks.


In [ ]:
from pymhm import Equation, MultiscaleProblem, assemble
from pymhm.fem.scalar.triangle import scalar_operators
from examples.formulations.darcy import pressure_constraints
from examples.formulations.three_field import define_three_field, recover_three_field
from examples.formulations.original import solve_original
from examples.mh2m_campaign import source
from examples.mh2m_heterogeneous import OscillatoryCoefficient
from examples.mh2m_heterogeneous_norms import StructuredP1, difference

material = OscillatoryCoefficient(gamma=1.8, epsilon=1 / 14)
mesh = TriangleMesh.unit_square(4)
gamma = PressureTraceSpace.uniform(mesh, 1, 1)
dimensions = []
current_fields = {}
with threadpool_limits(1):
    for segments in (1, 2):
        conormal = SkeletonSpace(mesh, tuple(FaceSpace.uniform(0, segments) for _ in mesh.faces))
        order_results = []
        for order in (8, 10):
            result_definition = define_three_field(
                mesh,
                permeability=material,
                source=source,
                dirichlet=0.0,
                pressure_trace=gamma,
                flux_space=conormal,
                degree=1,
                local_refinement=16,
                quadrature_order=order,
            )
            result_system = assemble(result_definition.problem)
            generic = result_system.solve(
                constraints=pressure_constraints(result_definition, result_system)
            )
            result = recover_three_field(result_definition, result_system, generic)
            original = solve_original(result_system)
            np.testing.assert_allclose(generic.trace, original.trace, atol=1e-12, rtol=1e-10)
            for condensed, uncondensed in zip(generic.fields, original.fields, strict=True):
                np.testing.assert_allclose(condensed, uncondensed, atol=1e-12, rtol=1e-10)
            balance = float(np.max(np.abs(result.conservation_residuals())))
            assert balance < 1e-12
            vertices = np.concatenate([local.mesh.points[local.mesh.cells] for local in result.local])
            pressures = np.concatenate([
                values[local.mesh.cells]
                for local, values in zip(result.local, result.pressure, strict=True)
            ])
            order_results.append(StructuredP1.from_arrays(vertices, pressures))
            print({
                "Lambda_segments": segments,
                "assembly_order": order,
                "Gamma_total": result.trace_space.size,
                "Gamma_free": len(result.free_dofs),
                "macro_balance": balance,
                "original_coefficient_max": max(
                    float(np.max(np.abs(a - b)))
                    for a, b in zip(generic.fields, original.fields, strict=True)
                ),
            })
        print("Assembly quadrature increment", difference(order_results[1], order_results[0], material, 10))
        current_fields[segments] = order_results[1]
        dimensions.append((segments, result.trace_space.size, len(result.free_dofs)))
assert dimensions[0][1:] == dimensions[1][1:]


## Independently assembled conforming reference and its refinement

The global P1 conforming equations below use the same physical coefficient,
forcing and homogeneous Dirichlet data. They retain a complete global nodal
vector and have no local elimination or Gamma/Lambda interpolation. The public
volume forms assemble K and F; `MultiscaleProblem.from_global` expresses their
variational equation with explicitly fixed boundary nodes.

The sequence has 64, 128 and 256 squares per axis and assembly order 8. Consecutive
pressure, raw Darcy flux and energy increments use the finer field's own norm
in their denominator. These increments measure the reference's remaining
resolution error; the finest field is a numerical comparison, not an exact
solution. The code prints these controls before reporting the coarse MH²M
comparisons. The executed 128-to-256 increment is approximately 1.41% in pressure and
12.20% in raw Darcy flux, so this short reference sequence remains unresolved
for a precision claim. These values limit the interpretation of every coarse
MH²M comparison below.
Common-overlay integration resolves both fine partitions and retains K(x), with
norm orders 8 and 10 reported separately. No convergence rate is inferred for
this oscillatory material.


In [ ]:
references = []
with threadpool_limits(1):
    for resolution in (64, 128, 256):
        reference_mesh = TriangleMesh.unit_square(resolution)
        matrix, mass, load = scalar_operators(
            reference_mesh, 1, diffusion=material, source=source, order=8
        )
        fixed = {
            int(node): 0.0
            for node in np.unique(reference_mesh.faces[reference_mesh.boundary_faces])
        }
        reference_problem = MultiscaleProblem.from_global(
            Equation(matrix, load), len(load), fixed=fixed
        )
        reference_system = assemble(reference_problem)
        reference_solution = reference_system.solve()
        reference = StructuredP1.from_arrays(
            reference_mesh.points[reference_mesh.cells],
            reference_solution.trace[reference_mesh.cells],
        )
        references.append(reference)
        print({"conforming_resolution": resolution, "dofs": len(load),
               "algebraic_residual": reference_solution.residual})
        if len(references) > 1:
            for order in (8, 10):
                print("Reference refinement increment", difference(reference, references[-2], material, order))
    for segments, field in current_fields.items():
        for order in (8, 10):
            print({"MH2M_Lambda_segments": segments, "reference_resolution": 256,
                   "physical_differences": difference(references[-1], field, material, order)})


## Current fields on the declared macro partition

All panels overlay the actual MH²M macroedges. The MH²M pressure uses independent
fine-cell vertices, so incident macrocell values are not averaged. The classical
pressure is its own continuous P1 field. The last panel shows the magnitude of
the raw physical Darcy flux -K grad(p) at fine-cell centroids; it is distinct from
the outward conormal coefficients and from an H(div) reconstruction. These
visual samples do not replace the volume norms above.


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.tri as mtri
from matplotlib.collections import LineCollection

vertices = np.concatenate([local.mesh.points[local.mesh.cells] for local in result.local])
pressures = np.concatenate([
    values[local.mesh.cells]
    for local, values in zip(result.local, result.pressure, strict=True)
])
points = vertices.reshape(-1, 2)
triangles = np.arange(len(points)).reshape(-1, 3)
triangulation = mtri.Triangulation(points[:, 0], points[:, 1], triangles)
centers = vertices.mean(axis=1)
_, gradients = current_fields[2].evaluate(centers)
flux_magnitude = material(centers) * np.linalg.norm(gradients, axis=1)
reference_triangulation = mtri.Triangulation(
    reference_mesh.points[:, 0], reference_mesh.points[:, 1], reference_mesh.cells
)
fig, axes = plt.subplots(2, 2, figsize=(10, 8), layout="constrained")
images = [
    axes[0, 0].tripcolor(triangulation, facecolors=material(centers), shading="flat"),
    axes[0, 1].tripcolor(triangulation, pressures.ravel(), shading="gouraud"),
    axes[1, 0].tripcolor(reference_triangulation, reference_solution.trace, shading="gouraud"),
    axes[1, 1].tripcolor(triangulation, facecolors=flux_magnitude, shading="flat"),
]
titles = ["Permeability K at fine-cell centroids", "MH²M pressure, Lambda segments=2",
          "Conforming P1 pressure, n=256", "MH²M raw Darcy flux magnitude at centroids"]
for axis, image, title in zip(axes.ravel(), images, titles, strict=True):
    axis.add_collection(LineCollection(mesh.points[mesh.faces], colors="black", linewidths=0.45))
    axis.set(title=title, xlabel="x", ylabel="y", aspect="equal", xlim=(0, 1), ylim=(0, 1))
    fig.colorbar(image, ax=axis, shrink=0.85)
plt.show()


## Independently refined classical P3 reference

DOLFINx/UFL assembles the same coefficient, source and boundary conditions on five SW–NE meshes. Consecutive differences use the finer field in each denominator. The final P3 n=512 flux increment is 0.0783701%; this is a refinement measurement, not an exact-error bound. Assembly orders 12 and 16 differ by about 2.39e-12 in relative flux. The archived-field energy agrees with a native UFL integral to about 1.99e-13 relative.

The 24 crisscross cases and 18 diagonal-mesh controls use this same reference for physical norms. Orders 8 and 10 are retained separately. The article's P1 n=128 curve remains unchanged in the published-profile overlays.


In [ ]:
if HISTORICAL_REPLAY:
    cg3_root = ROOT / "examples/results/mh2m-heterogeneous/cg3"
    cg3 = json.loads((cg3_root / "comparison.json").read_text())
    cg3_controls = json.loads((cg3_root / "structured-comparison.json").read_text())
    cg3_reference = cg3["reference_acquisitions"][-1]
    assert len(cg3["reference_acquisitions"]) == 5 and len(cg3["increments"]) == 4
    assert cg3_controls["reference_acquisition"] == cg3_reference
    for acquisition in [*cg3["reference_acquisitions"],
                        cg3["assembly_quadrature_control"]["acquisition"]]:
        assert not acquisition["source_changed_during_run"]
        assert hashlib.sha256((cg3_root / acquisition["archive"]).read_bytes()).hexdigest() == acquisition["archive_sha256"]
    for previous, current, row in zip(cg3["reference_acquisitions"][:-1],
                                      cg3["reference_acquisitions"][1:],
                                      cg3["increments"], strict=True):
        assert row["reference_archive"] == current["archive"]
        assert row["reference_sha256"] == current["archive_sha256"]
        assert row["other_archive"] == previous["archive"]
        assert row["other_sha256"] == previous["archive_sha256"]
        print(row["name"], row["norms"]["quadrature_10"])
    print("Assembly quadrature control", cg3["assembly_quadrature_control"]["norms"])
    print("P1 n=1024 versus P3 n=512", cg3["cg1_reference_comparison"]["norms"])
    print(json.loads((cg3_root / "native-norm-verification.json").read_text()))


## Diagonal-mesh control series

These records use uniform triangular subdivisions with southwest-to-northeast diagonals. The pressure, broken gradient, raw physical flux and energy differences are integrated on a common subdivision resolving both triangle meshes. Pressure is not transferred to only one grid. The primary physical comparisons below use the independent continuous P3 n=512 field. The original P1 refinement series remains a separate reference-resolution control. Every norm uses its stated reference denominator; no percentage from one baseline is combined with another.


In [ ]:
if HISTORICAL_REPLAY:
    record = json.loads((ROOT / "examples/results/mh2m-heterogeneous/comparison.json").read_text())
    assert not record["source_changed_during_run"]
    print(record["conventions"])
    for row in record["references"]:
        if "increment" in row:
            print("Separate classical P1 control", row["resolution"], row["increment"])


In [ ]:
if HISTORICAL_REPLAY:
    structured_norms = {row["archive"]: row for row in cg3_controls["cases"]}
    assert len(structured_norms) == len(record["cases"]) == 18
    for row in record["cases"]:
        comparison = structured_norms[row["archive"]]
        archive = ROOT / "examples/results/mh2m-heterogeneous" / row["archive"]
        assert hashlib.sha256(archive.read_bytes()).hexdigest() == row["archive_sha256"]
        assert row["archive_sha256"] == comparison["archive_sha256"]
        print(row["method"], row["family"], row["macro_resolution"],
              row["local_refinement"], row["Lambda_segments"], row["global_dofs_free"],
              "versus P3 n=512", comparison["norms"]["quadrature_10"])


## Diagonal-mesh fields and interface enrichment

The fixed-Gamma series varies the conormal segments without increasing the global pressure-trace dimension. The diagonal-mesh spatial control uses n=31, local refinement 4, and two segments for Gamma and Lambda. This construction is distinct from the recovered crisscross connectivity below. One-sided profile segments and actual macro boundaries are retained. Raw flux maps are distinct from conormal moments and H(div) reconstructions.


In [ ]:
if HISTORICAL_REPLAY:
    for name in ("refinement", "fixed-gamma", "profiles", "fields"):
        display(Image(filename=str(ROOT / f"docs/figures/mh2m-heterogeneous/{name}.png")))


## Recovered crisscross connectivity

Each macro has 2r² fine triangles and r,r,2r boundary edges. The local pairing with eight constant conormal segments per macroedge has rank 15 of 23 at r=4, and full rank 23 at r=8. A zero-mean Neumann map cannot invert the former pair. The recorded campaign uses admissible pairs and keeps the printed Figure 6 integer dimensions separate from its inconsistent subdivision label. The geometric maximum diameters are Hmax=sqrt(2)/n and hmax=1/(n*r), while the dissertation's Cartesian spacing is 1/n.

The Figure 5 configuration uses n=31, r=4 and two Gamma/Lambda segments. Both incident profile values are preserved. Its pressure profiles retain the printed P1 n=128 resolution; physical differences use the independently refined P3 n=512 field. Visual proximity alone does not establish historical reproduction.

The additional asinh view changes only the flux color normalization. Reference and MH²M share physical limits and linear widths; differences have their own limits, and all ticks retain physical values. Pressure, fields and norms are unchanged.

Figure-6 controls retain the printed global dimensions separately from the caption: four uniform subfaces give the lower-row dimensions, while eight conormal segments provide an explicit local enrichment at fixed Gamma. The upper starred r=4/sLambda=8 pair fails the local injectivity condition.


In [ ]:
if HISTORICAL_REPLAY:
    cross_root = ROOT / "examples/results/mh2m-heterogeneous/crisscross"
    cross = json.loads((cross_root / "comparison.json").read_text())
    assert not cross["source_changed_during_run"]
    cross_norms = {row["name"]: row for row in cg3["cases"]}
    assert len(cross_norms) == len(cross["cases"]) == 24
    print(cross["conventions"])
    for row in cross["cases"]:
        assert hashlib.sha256((cross_root / row["archive"]).read_bytes()).hexdigest() == row["archive_sha256"]
        comparison = cross_norms[row["name"]]
        assert comparison["archive_sha256"] == row["archive_sha256"]
        assert comparison["reference_sha256"] == cg3_reference["archive_sha256"]
        print(row["name"], row["global_dofs_total"], row["global_dofs_free"],
              "versus P3 n=512", comparison["norms"]["quadrature_10"])
    print(json.loads((cross_root / "space-compatibility.json").read_text())["rows"])


In [ ]:
if HISTORICAL_REPLAY:
    for name in ("profiles", "fields", "fields-asinh", "refinement", "fixed-gamma",
                 "figure-6-upper-profiles", "figure-6-lower-profiles",
                 "published-figure-7-comparison", "published-figure-8-comparison"):
        path = ROOT / f"docs/figures/mh2m-heterogeneous/crisscross/{name}.png"
        display(Image(filename=str(path)))


The printed-profile comparisons use the original embedded raster colours and retain their visible line spans and one-pixel uncertainty. Occluded colours are not interpolated. Both incident numerical traces are shown without amplitude scaling. These profile comparisons are distinct from volume norms, and disagreement beyond raster uncertainty is retained.


In [ ]:
if HISTORICAL_REPLAY:
    printed = json.loads((cross_root / "published-profile-comparison.json").read_text())
    print(printed["comparison"])
    for row in printed["rows"]:
        print(row["name"], row["estimates"])


The classical reference profile can also be checked independently of MH²M. The following controls retain the same coefficient, source and 128 Cartesian subdivisions, comparing northwest–southeast and crisscross connectivities at assembly orders 10 and 14. Crisscross has twice as many triangles; neither control replaces the independently refined reference series. Differences below are evaluated on visible raster columns and are not volume norms.

In [ ]:
if HISTORICAL_REPLAY:
    geometry = json.loads((cross_root / "reference-geometry/comparison.json").read_text())
    assert not geometry["source_changed_during_run"]
    for row in geometry["cases"]:
        print(row["geometry"], row["quadrature_order"], row["dofs"], row["profiles"])
        if "previous_order_pressure_difference_l2_relative" in row:
            print("Quadrature pressure L2 change:", row["previous_order_pressure_difference_l2_relative"])

## Additional independent P1 verification

The independent P1 verification uses DOLFINx/UFL to assemble the six original n=32,...,1024 operators with the same physical Duffy-8 rule. It also includes a nonhomogeneous boundary patch. These checks distinguish the implemented classical field from identification of the historical figure.

The recorded n=128/130 and epsilon=1/14 or 1/17 controls, and nine fixed quadrature rules, do not reconcile all visible reference-curve columns in Figures 7 and 8. Raster envelopes include one pixel and the visible line thickness; no amplitude normalization is applied. These controls do not identify the unique historical data or geometry.

In [ ]:
if HISTORICAL_REPLAY:
    native_p1 = json.loads((ROOT / "examples/results/mh2m-heterogeneous/cg1-native-verification.json").read_text())
    assert not native_p1["source_changed_during_comparison"]
    print(native_p1["scope"])
    print(native_p1["nonhomogeneous_affine_patch"])
    print(native_p1["fields"])
    print(native_p1["documented_data_controls"])
    print(native_p1["quadrature_controls"])